[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Cascades and Deletes


## What you will be able to do

Decide, relationship by relationship, what deleting a parent does to the rows that point at it: keep
them with their foreign key set to `NULL`, delete them with it through
`cascade="all, delete-orphan"`, or leave them to the database with `passive_deletes=True` and
`ondelete="CASCADE"`. Delete a child by taking it out of its collection, delete rows chosen by a
condition with one `delete()` statement, and recognize the errors from a key the default cannot set
to `NULL`, a bulk delete that skipped the ORM's rules, a cascade on the wrong side, and a row
another program deleted first.


## The idea

### The problem

The registrar's office deletes things. An advisor leaves the college, and the advisor's students
stay, waiting for someone new. A student admitted for the autumn withdraws before classes start, and
the record goes, with the three courses it signed up for. A section is canceled, and nobody can stay
enrolled in a section that does not exist. A term planned too early comes off the calendar, with all
of its sections at once.

Every one of these deletes a row that other rows point at, and every one wants something different
done with them. The ORM does one thing unless it is told otherwise: it keeps the other rows and sets
their foreign key to `NULL`. That is right for the advisor's students. It cannot work for a section,
whose course may not be `NULL`. A delete that does work can be slow, since the ORM loads every child
before it deletes it. And a delete written as one SQL statement skips the ORM's rules altogether.

### What cascades are

> A relationship's **cascade** lists what the session does to the related objects when it does
> something to this one. The default, `"save-update, merge"`, adds new children to the session with
> their parent, and when the parent is deleted, sets their foreign key to `NULL`. **`delete`**
> deletes the children with their parent, and **`delete-orphan`** also deletes a child taken out of
> the collection; `cascade="all, delete-orphan"` is the usual way to ask for both.
> **`ondelete="CASCADE"`**, on a `ForeignKey`, is the database's own rule, and
> **`passive_deletes=True`** tells the ORM to rely on it instead of loading the children. A
> **bulk delete**, `session.execute(delete(Enrollment).where(...))`, is one statement, and runs none
> of the ORM's cascades.

### Why it works that way

- **The ORM can only change rows it has loaded.** To set a child's key to `NULL` or delete it, the
  session needs the child as an object, so it sends a `SELECT` for the children first.
- **The database can change rows nobody loaded.** `ON DELETE CASCADE` deletes the rows that point at
  a deleted row as part of the one `DELETE`, which SQLite does only on a connection that switched
  foreign keys on, as `college_engine` does.
- **A key that must have a value cannot be set to `NULL`.** For a child that cannot exist without
  its parent, the choice is between deleting it with the parent and refusing to delete the parent.
- **A cascade belongs to the parent's side.** It says what happens to the objects in a collection,
  so it goes on the one-to-many relationship, not on the child's reference to its parent.
- **A bulk delete never sees the objects.** It is SQL sent as written, so only the database's rules
  apply to the rows it deletes.
- **A row can go while a session holds its object.** Another program can delete it in between, and
  the session finds out only when its own `UPDATE` or `DELETE` matches no row.

### Where this shows up

Every application that deletes anything decides these rules, often without meaning to. The
**Constraints** notebook of the **sqlite3, Deep Dive** guide wrote `ON DELETE CASCADE` in SQL, and
showed that SQLite enforces foreign keys only on a connection that switched them on. **The Session**
notebook deleted one object at a time with `session.delete()`, and **The Identity Map** notebook met
a row another program had deleted, from the side of a read. **Migrations with Alembic** changes the
constraints of a table that already has rows in it.

### What this notebook covers

- Fall 2026, planned: rows to delete
- The default: children kept, with their key set to `NULL`
- `cascade="all, delete-orphan"`: children deleted with their parent
- `delete-orphan`: a child taken out of its collection
- `passive_deletes=True`: the database deletes the children
- A bulk `delete()`: one statement, and no cascades
- Which rule to declare for which relationship
- Canceling a planned term, finished
- Four errors, from a key the default cannot set to `NULL` to a row another program deleted first

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import ForeignKey, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Advisor(Base):
    __tablename__ = "advisors"
    id: Mapped[int] = mapped_column(primary_key=True)
    students: Mapped[list["Student"]] = relationship()                     # the default rule


class Student(Base):
    __tablename__ = "students"
    id: Mapped[int] = mapped_column(primary_key=True)
    advisor_id: Mapped[int | None] = mapped_column(ForeignKey("advisors.id"))
    notes: Mapped[list["Note"]] = relationship(cascade="all, delete-orphan")


class Note(Base):
    __tablename__ = "notes"
    id: Mapped[int] = mapped_column(primary_key=True)
    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"))


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    first, second = Student(notes=[Note(), Note()]), Student(notes=[Note()])
    advisor = Advisor(students=[first, second])
    session.add(advisor)
    session.commit()
    session.delete(advisor)
    session.delete(first)
    session.commit()
    print("students:", session.execute(select(Student.id, Student.advisor_id)).all())
    print("notes:   ", session.execute(select(Note.id, Note.student_id)).all())
```

```
students: [(2, None)]
notes:    [(3, 2)]
```

Deleting the advisor kept both students and set their `advisor_id` to `None`, which is what a
relationship does unless it is told otherwise. Deleting the first student, whose `notes` has
`cascade="all, delete-orphan"`, deleted that student's two notes as well, and the second student's
note stayed where it was.


## Setup

Ten imports, and the college built from its classes, with the rules for deleting that this notebook
is about.

- `sqlalchemy` is the library itself, and the cell prints its version
- `delete`, from `sqlalchemy`, writes a bulk delete, with `select`, `func`, `insert`, `create_engine`
  and `event`, and what the classes need
- `relationship`, from `sqlalchemy.orm`, takes the `cascade` and `passive_deletes` options, with the
  rest of the ORM
- `StaleDataError`, from `sqlalchemy.orm.exc`, is the error the last of the Common errors catches,
  and `warnings` catches the warning that goes with it
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup's classes are those of the **Relationships** notebook, with a class for advisors and a rule for
deleting on four of the relationships:

- `Advisor` is new, and `Student.advisor_id` refers to it and may be `NULL`. `Advisor.students` keeps
  the default rule.
- `Student.enrollments` has `cascade="all, delete-orphan"`.
- `Term.sections` and `Section.enrollments` have `cascade="all, delete-orphan"` and
  `passive_deletes=True`, and the foreign keys they follow, `sections.term_id` and
  `enrollments.section_id`, have `ondelete="CASCADE"`.
- `Course.sections` keeps the default rule.

The advisors' table starts empty, and the first worked example fills it.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, delete, event, func,
                        insert, select)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker
from sqlalchemy.orm.exc import StaleDataError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Advisor(Base):
    __tablename__ = "advisors"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))

    students: Mapped[list["Student"]] = relationship(back_populates="advisor", order_by="Student.id")

    def __repr__(self):
        return f"Advisor({self.name!r})"


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]
    advisor_id: Mapped[int | None] = mapped_column(ForeignKey("advisors.id"))

    advisor: Mapped[Advisor | None] = relationship(back_populates="students")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id",
                                                           cascade="all, delete-orphan")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id",
                                                     cascade="all, delete-orphan", passive_deletes=True)

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id", ondelete="CASCADE"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id",
                                                           cascade="all, delete-orphan", passive_deletes=True)

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id", ondelete="CASCADE"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### Fall 2026, planned: rows to delete

A notebook about deleting needs rows it can delete without taking the college's history with them.
So the registrar gives every program an advisor, opens Fall 2026 with a section of every course,
admits Zoe Nakamura for the autumn, and signs every student up for three Fall 2026 courses:


In [2]:
ADVISORS = {"Biology": "Miriam Hale", "Computer Science": "Samuel Osei", "Mathematics": "Lucia Ferrante",
            "Psychology": "Anders Lund", "History": "Nadia Rahman"}

with SessionLocal.begin() as session:
    advisors = {program: Advisor(name=name) for program, name in ADVISORS.items()}
    for student in session.scalars(select(Student)):
        student.advisor = advisors[student.program]                 # one advisor for every program
    fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
    session.add(fall)
    for course in session.scalars(select(Course).order_by(Course.id)):
        fall.sections.append(Section(course=course, capacity=30))   # sections 41 to 50
    session.add(Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                        started_on=date(2026, 8, 24), advisor=advisors["Computer Science"]))
    for student in session.scalars(select(Student).order_by(Student.id)):
        for step in (0, 3, 6):                                      # three courses each, as in every term
            session.add(Enrollment(student=student, section=fall.sections[(student.id + step) % 10]))


def counts():
    """The number of rows in every table this notebook deletes from."""
    with SessionLocal() as session:
        return {cls.__tablename__: session.scalar(select(func.count()).select_from(cls))
                for cls in (Advisor, Student, Term, Section, Enrollment)}


print(counts())


{'advisors': 5, 'students': 26, 'terms': 5, 'sections': 50, 'enrollments': 306}


Five advisors; a fifth term, Fall 2026, with sections 41 to 50; a twenty-sixth student; and 78 new
enrollments, three for each of the twenty-six. `counts()` reports every table this notebook deletes
from, and every delete in the worked examples falls on these new rows.

### The default: children kept, with their key set to NULL

Nadia Rahman, the History advisor, leaves the college. `Advisor.students` was declared with no rule
of its own, so it has the default, which the first line prints:


In [3]:
print(Advisor.students.property.cascade)

engine.echo = True
with SessionLocal() as session:
    nadia = session.scalars(select(Advisor).where(Advisor.name == "Nadia Rahman")).one()
    session.delete(nadia)
    session.commit()
engine.echo = False

with SessionLocal() as session:
    print(session.execute(select(Student.name, Student.advisor_id).where(Student.program == "History")).all())
print(counts())


CascadeOptions('merge,save-update')
    BEGIN (implicit)
    SELECT advisors.id, advisors.name
    FROM advisors
    WHERE advisors.name = ?
    values: ('Nadia Rahman',)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on, students.advisor_id AS students_advisor_id
    FROM students
    WHERE ? = students.advisor_id ORDER BY students.id
    values: (5,)
    UPDATE students SET advisor_id=? WHERE students.id = ?
    values: [(None, 5), (None, 10), (None, 15), (None, 20), (None, 25)]
    DELETE FROM advisors WHERE advisors.id = ?
    values: (5,)
    COMMIT
[('Elena Petrova', None), ('Jonas Berg', None), ('Olivia Brandt', None), ('Tara Nilsen', None), ("Aoife O'Brien", None)]
{'advisors': 4, 'students': 26, 'terms': 5, 'sections': 50, 'enrollments': 306}


The session loaded the advisor, then the advisor's students with a `SELECT` of their own, and set
every one's `advisor_id` to `NULL` in one `UPDATE` with five sets of values, before the `DELETE`. The
five History students are all still there, with no advisor, which is what should happen to a student
whose advisor leaves. The second `SELECT` is the cost of the default: the ORM changes only rows it
has loaded, so it loads them.

### cascade="all, delete-orphan": children deleted with their parent

Zoe Nakamura withdraws before the autumn begins, and the record goes, with its three Fall 2026
enrollments. `Student.enrollments` is declared with `cascade="all, delete-orphan"`, and the first
line prints what that stands for:


In [4]:
print(Student.enrollments.property.cascade)

engine.echo = True
with SessionLocal() as session:
    zoe = session.scalars(select(Student).where(Student.name == "Zoe Nakamura")).one()
    session.delete(zoe)
    session.commit()
engine.echo = False
print(counts())


CascadeOptions('delete,delete-orphan,expunge,merge,refresh-expire,save-update')
    BEGIN (implicit)
    SELECT students.id, students.name, students.email, students.program, students.started_on, students.advisor_id
    FROM students
    WHERE students.name = ?
    values: ('Zoe Nakamura',)
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE ? = enrollments.student_id ORDER BY enrollments.section_id
    values: (26,)
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: [(26, 43), (26, 47), (26, 50)]
    DELETE FROM students WHERE students.id = ?
    values: (26,)
    COMMIT
{'advisors': 4, 'students': 25, 'terms': 5, 'sections': 50, 'enrollments': 303}


`all` stood for three cascades besides the default's two, `delete` among them, and `delete-orphan`
was named beside it. The session loaded Zoe Nakamura's enrollments, deleted them with one `DELETE`
sent with three sets of values, and deleted the student last, so that no enrollment ever pointed at
a student that was gone.

### delete-orphan: a child taken out of its collection

Chloe Martin drops Composition for Fall 2026 before the term begins. With `delete-orphan`, taking the
enrollment out of `chloe.enrollments` is all it takes:


In [5]:
engine.echo = True
with SessionLocal() as session:
    chloe = session.get(Student, 3)
    composition = next(enrollment for enrollment in chloe.enrollments if enrollment.section_id == 47)
    chloe.enrollments.remove(composition)
    session.commit()
engine.echo = False
print(counts())


    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on, students.advisor_id AS students_advisor_id
    FROM students
    WHERE students.id = ?
    values: (3,)
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE ? = enrollments.student_id ORDER BY enrollments.section_id
    values: (3,)
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: (3, 47)
    COMMIT
{'advisors': 4, 'students': 25, 'terms': 5, 'sections': 50, 'enrollments': 302}


There is no `session.delete()` in the cell. Taken out of the collection, the enrollment belonged to
no student, which is what an orphan is, and `delete-orphan` deleted it at the flush. Dropping a
course this way suits a term that has not begun. Once a term is under way, the college keeps the
enrollment and changes its status to `'withdrawn'`, which is an `UPDATE` and not a delete.

### passive_deletes=True: the database deletes the children

Statistics is canceled for Fall 2026. `Section.enrollments` has `passive_deletes=True` beside its
cascade, and the foreign key from `enrollments` to `sections` carries the database's own rule, which
the first two lines read back from the database:


In [6]:
for key in sqlalchemy.inspect(engine).get_foreign_keys("enrollments"):
    print(key["constrained_columns"], "refers to", key["referred_table"], key["options"])

with SessionLocal() as session:
    print("enrollments in section 50:", session.scalar(select(func.count()).where(Enrollment.section_id == 50)))

engine.echo = True
with SessionLocal() as session:
    session.delete(session.get(Section, 50))                        # Statistics, Fall 2026
    session.commit()
engine.echo = False

with SessionLocal() as session:
    print("enrollments in section 50:", session.scalar(select(func.count()).where(Enrollment.section_id == 50)))
print(counts())


['student_id'] refers to students {}
['section_id'] refers to sections {'ondelete': 'CASCADE'}
enrollments in section 50: 7
    BEGIN (implicit)
    SELECT sections.id AS sections_id, sections.course_id AS sections_course_id, sections.term_id AS sections_term_id, sections.capacity AS sections_capacity
    FROM sections
    WHERE sections.id = ?
    values: (50,)
    DELETE FROM sections WHERE sections.id = ?
    values: (50,)
    COMMIT
enrollments in section 50: 0
{'advisors': 4, 'students': 25, 'terms': 5, 'sections': 49, 'enrollments': 295}


One `SELECT` for the section, one `DELETE`, and seven enrollments gone. The database deleted them
itself, as part of the `DELETE`, because the foreign key says `ON DELETE CASCADE`, and
`passive_deletes=True` is what kept the session from loading them first, as it loaded Zoe Nakamura's
three. The rule works only on a connection that switched foreign keys on, which `college_engine`
does for every connection, as the **Engines and URLs** notebook showed.

### A bulk delete(): one statement, and no cascades

The Psychology students' Fall 2026 registrations are cleared, to be done again after an advising
day. That needs no objects, only a condition, and `delete()` sent with `session.execute` is one
statement:


In [7]:
FALL_2026 = select(Section.id).join(Section.term).where(Term.name == "Fall 2026")
PSYCHOLOGY = select(Student.id).where(Student.program == "Psychology")
CLEAR_PSYCHOLOGY = delete(Enrollment).where(Enrollment.section_id.in_(FALL_2026), Enrollment.student_id.in_(PSYCHOLOGY))

print(" ".join(str(CLEAR_PSYCHOLOGY.compile(engine)).split()))
with SessionLocal() as session:
    result = session.execute(CLEAR_PSYCHOLOGY)
    session.commit()
print(result.rowcount, "enrollments deleted")
print(counts())


DELETE FROM enrollments WHERE enrollments.section_id IN (SELECT sections.id FROM sections JOIN terms ON terms.id = sections.term_id WHERE terms.name = ?) AND enrollments.student_id IN (SELECT students.id FROM students WHERE students.program = ?)
13 enrollments deleted
{'advisors': 4, 'students': 25, 'terms': 5, 'sections': 49, 'enrollments': 282}


Thirteen enrollments in one statement, and not one object loaded: the two conditions are subqueries
that the database ran itself, and `rowcount` is how many rows the statement deleted. A bulk delete is
SQL sent as written, so none of the ORM's rules run on it, and only the database's own, its foreign
keys and their `ON DELETE`, apply. Nothing points at an enrollment, so deleting enrollments this way
is safe. Deleting advisors this way is not, which is the second of the Common errors.

### Which rule to declare for which relationship

Every one-to-many relationship answers one question: what should the children do when their parent
is deleted?

| When the parent is deleted, the children should | Declare on the parent's relationship | What the session sends |
|---|---|---|
| stay, with no parent | nothing: the default, over a foreign key that may be `NULL` | a `SELECT` of the children, an `UPDATE` of their key, then the parent's `DELETE` |
| go with it | `cascade="all, delete-orphan"` | a `SELECT` of the children, a `DELETE` of them, then the parent's `DELETE` |
| go with it, and there may be many | `cascade="all, delete-orphan", passive_deletes=True`, and `ondelete="CASCADE"` on the foreign key | the parent's `DELETE`, and the database deletes the children |
| keep the parent from being deleted | the default, and a check before `session.delete()` that says why | nothing, until the check passes |
| outlast the parent as a record | no delete at all, but a status, such as `'withdrawn'` | an `UPDATE` of the status |

For children that belong to their parent, `cascade="all, delete-orphan"` is the one to reach for
first, and `passive_deletes=True` with `ondelete="CASCADE"` is worth adding once a parent can have
many children. `session.execute(delete(...))` is for rows chosen by a condition, and runs none of
these rules.

### Canceling a planned term, finished

The pieces of this notebook in one function. `cancel_term` takes a term off the calendar with its
sections and their enrollments, and refuses a term that has started, whose enrollments are the
college's record of what happened. `today` is an argument rather than the clock's date, so the
function answers the same way on every run:


In [8]:
def cancel_term(session, name, today):
    """Take a term that has not started off the calendar, with its sections and their enrollments, which the
    database deletes. A term that has started is kept, since its enrollments are the record of what happened."""
    term = session.scalars(select(Term).where(Term.name == name)).one()
    if term.starts_on <= today:
        raise ValueError(f"{name} started on {term.starts_on}, and a term that has started is kept")
    sections = session.scalar(select(func.count()).select_from(Section).where(Section.term_id == term.id))
    enrollments = session.scalar(
        select(func.count()).select_from(Enrollment).join(Enrollment.section).where(Section.term_id == term.id)
    )
    session.delete(term)
    return sections, enrollments


TODAY = date(2026, 4, 15)                                           # a day in Spring 2026, which is under way

try:
    with SessionLocal.begin() as session:
        cancel_term(session, "Spring 2026", TODAY)
except ValueError as error:
    print("refused:", error)

engine.echo = True
with SessionLocal.begin() as session:
    sections, enrollments = cancel_term(session, "Fall 2026", TODAY)
engine.echo = False
print(f"canceled Fall 2026, and with it {sections} sections and {enrollments} enrollments")
print(counts())


refused: Spring 2026 started on 2026-01-12, and a term that has started is kept
    BEGIN (implicit)
    SELECT terms.id, terms.name, terms.starts_on
    FROM terms
    WHERE terms.name = ?
    values: ('Fall 2026',)
    SELECT count(*) AS count_1
    FROM sections
    WHERE sections.term_id = ?
    values: (5,)
    SELECT count(*) AS count_1
    FROM enrollments JOIN sections ON sections.id = enrollments.section_id
    WHERE sections.term_id = ?
    values: (5,)
    DELETE FROM terms WHERE terms.id = ?
    values: (5,)
    COMMIT
canceled Fall 2026, and with it 9 sections and 54 enrollments
{'advisors': 4, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


Spring 2026 was refused before anything was deleted. Fall 2026 went with one `DELETE`, after the
`SELECT`s that found and counted: the database deleted the term's sections, because
`sections.term_id` says `ON DELETE CASCADE`, and their enrollments, because `enrollments.section_id`
says it too, one cascade setting off the next. The counts are Setup's again, but for the four
advisors.

### Where each part came from

| In `cancel_term` | What it relies on | The section that showed it |
|---|---|---|
| `session.delete(term)`, with no `SELECT` of the sections | `passive_deletes=True` on `Term.sections` | passive_deletes=True: the database deletes the children |
| the sections and their enrollments gone with the term | `ondelete="CASCADE"` on both foreign keys, on a connection with foreign keys on | passive_deletes=True: the database deletes the children |
| the counts taken before the `DELETE` | children the database deletes unseen, which only a count taken first can report | passive_deletes=True: the database deletes the children |
| a term that has started refused | a record kept rather than deleted | Which rule to declare for which relationship |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/15-cascades-and-deletes-solutions.ipynb).

**1.** Delete the advisor Anders Lund with `echo` on, and print the names of the students left with
no advisor.


In [9]:
# your code here


**2.** Admit Priya Shah to Psychology, starting on 2026-08-24, with enrollments in two Fall 2026
sections. Then delete the new student with `echo` on, and count the enrollments before and after.


In [10]:
# your code here


**3.** Daniel Kim, student 4, drops World History for Fall 2026. Take that enrollment out of
`enrollments` with `echo` on, and print the Fall 2026 sections Daniel Kim still has.


In [11]:
# your code here


**4.** Cancel the Fall 2026 section of Data Structures with `echo` on, and count its enrollments
before and after.


In [12]:
# your code here


**5.** With one bulk `delete()`, remove every Fall 2026 enrollment in the Mathematics department's
courses, and print how many rows it deleted.


In [13]:
# your code here


**6.** Load the Fall 2026 section of Composition with its enrollments, using `selectinload`, and
delete it with `echo` on. Which deleted the enrollments this time, the session or the database?


In [14]:
# your code here


## Common errors

### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: sections.course_id


In [15]:
with SessionLocal() as session:
    history = session.scalars(select(Course).where(Course.code == "HIS-110")).one()
    session.delete(history)
    session.commit()


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: sections.course_id
[SQL: UPDATE sections SET course_id=? WHERE sections.id = ?]
[parameters: [(None, 8), (None, 18), (None, 28), (None, 38)]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

`Course.sections` keeps the default rule, so deleting World History meant setting `course_id` to
`NULL` in its four sections, and the column is `NOT NULL`. **The Session** notebook met the same
message from a section made without a course; here it comes from the rule. The database's refusal
happens to be right, since a course with sections has students' grades hanging from it, but a
refusal that reads like a bug is a poor way to say so. Decide the rule, and write it where the next
reader will find it:


In [16]:
def delete_course(session, code):
    """Delete a course that has never had a section. A course with sections has a history, and is kept."""
    course = session.scalars(select(Course).where(Course.code == code)).one()
    if course.sections:
        raise ValueError(f"{code} has {len(course.sections)} sections, and a course with sections is kept")
    session.delete(course)


with SessionLocal.begin() as session:
    session.add(Course(code="ART-100", title="Drawing I", department="Art", credits=3))

for code in ("HIS-110", "ART-100"):
    try:
        with SessionLocal.begin() as session:
            delete_course(session, code)
        print("deleted", code)
    except ValueError as error:
        print("refused:", error)


refused: HIS-110 has 4 sections, and a course with sections is kept
deleted ART-100


### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed


In [17]:
with SessionLocal() as session:
    session.execute(delete(Advisor).where(Advisor.name == "Samuel Osei"))
    session.commit()


IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: DELETE FROM advisors WHERE advisors.name = ?]
[parameters: ('Samuel Osei',)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

A bulk `delete()` went straight to the database, and the default rule of `Advisor.students`, which
would have set the Computer Science students' `advisor_id` to `NULL` first, never ran. The database
found students pointing at the advisor, had no `ON DELETE` rule for them, and refused. To have the
ORM's rules run, delete the object:


In [18]:
engine.echo = True
with SessionLocal() as session:
    samuel = session.scalars(select(Advisor).where(Advisor.name == "Samuel Osei")).one()
    session.delete(samuel)
    session.commit()
engine.echo = False
print(counts())


    BEGIN (implicit)
    SELECT advisors.id, advisors.name
    FROM advisors
    WHERE advisors.name = ?
    values: ('Samuel Osei',)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on, students.advisor_id AS students_advisor_id
    FROM students
    WHERE ? = students.advisor_id ORDER BY students.id
    values: (2,)
    UPDATE students SET advisor_id=? WHERE students.id = ?
    values: [(None, 2), (None, 7), (None, 12), (None, 17), (None, 22)]
    DELETE FROM advisors WHERE advisors.id = ?
    values: (2,)
    COMMIT
{'advisors': 3, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


### sqlalchemy.exc.ArgumentError: For many-to-one relationship Member.club, delete-orphan cascade is normally configured only on the "one" side of a one-to-many relationship, and not on the "many" side of a many-to-one or many-to-many relationship.  To force this relationship to allow a particular "Club" object to be referenced by only a single "Member" object at a time via the Member.club relationship, which would allow delete-orphan cascade to take place in this direction, set the single_parent=True flag.


In [19]:
class ClubBase(DeclarativeBase):
    pass


class Club(ClubBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


class Member(ClubBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"))
    club: Mapped[Club] = relationship(cascade="all, delete-orphan")         # meant: delete the members with their club


Club(name="Chess")


ArgumentError: For many-to-one relationship Member.club, delete-orphan cascade is normally configured only on the "one" side of a one-to-many relationship, and not on the "many" side of a many-to-one or many-to-many relationship.  To force this relationship to allow a particular "Club" object to be referenced by only a single "Member" object at a time via the Member.club relationship, which would allow delete-orphan cascade to take place in this direction, set the single_parent=True flag. (Background on this error at: https://sqlalche.me/e/20/bbf0)

The cascade went on `Member.club`, the member's reference to its club, where it would mean: when a
member is deleted, delete the club too. SQLAlchemy refused when the classes were first used, since a
club has many members, and deleting it along with one of them would take it away from the rest. The
way out the error offers, `single_parent=True`, is for the rare child that owns its parent. A cascade
says what happens to the objects in a collection, so it goes on the collection:


In [20]:
class ClubBase(DeclarativeBase):
    pass


class Club(ClubBase):
    __tablename__ = "clubs"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    members: Mapped[list["Member"]] = relationship(back_populates="club", cascade="all, delete-orphan")


class Member(ClubBase):
    __tablename__ = "members"

    id: Mapped[int] = mapped_column(primary_key=True)
    club_id: Mapped[int] = mapped_column(ForeignKey("clubs.id"))
    club: Mapped[Club] = relationship(back_populates="members")


club_engine = college_engine()
ClubBase.metadata.create_all(club_engine)
ClubSession = sessionmaker(club_engine)
with ClubSession.begin() as session:
    session.add(Club(name="Chess", members=[Member(), Member()]))
with ClubSession.begin() as session:
    session.delete(session.scalars(select(Club)).one())
with ClubSession() as session:
    print("members left:", session.scalar(select(func.count()).select_from(Member)))
club_engine.dispose()


members left: 0


### sqlalchemy.orm.exc.StaleDataError: UPDATE statement on table 'students' expected to update 1 row(s); 0 were matched.


In [21]:
Keeping = sessionmaker(engine, expire_on_commit=False)             # keeps its values after a commit

with Keeping() as session:
    liam = session.get(Student, 12)                                 # Liam Murphy
    session.commit()                                                # the read ends, and liam keeps its values

    with engine.begin() as conn:                                    # another program deletes the student
        conn.execute(delete(Enrollment).where(Enrollment.student_id == 12))
        conn.execute(delete(Student).where(Student.id == 12))

    liam.email = "liam.murphy@college.edu"
    session.commit()


StaleDataError: UPDATE statement on table 'students' expected to update 1 row(s); 0 were matched.

The session kept Liam Murphy's values after the commit, which is what `expire_on_commit=False` is
for, and in between, another program deleted the row. The `UPDATE` matched no row, and the session
raised rather than report a change it had not saved. The same race on a delete only warns, since the
row is gone either way:


In [22]:
with Keeping() as session:
    enrollment = session.get(Enrollment, (13, 40))                  # Maya Patel in Statistics, Spring 2026
    session.commit()

    with engine.begin() as conn:                                    # another program deletes it first
        conn.execute(delete(Enrollment).where(Enrollment.student_id == 13, Enrollment.section_id == 40))

    session.delete(enrollment)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        session.commit()
for warning in caught:
    print(type(warning.message).__name__ + ":", warning.message)


The cell caught the warning and printed it, since a warning printed the ordinary way carries the
name of a temporary file. Neither the error nor the warning can be prevented from the session's side,
because the other program's delete comes first. Catch the error, roll back, and look for the row
again: `session.get` answers `None` for a row that is no longer there, as **The Identity Map**
notebook showed.


In [23]:
with Keeping() as session:
    wes = session.get(Student, 23)                                  # Wes Carter
    session.commit()

    with engine.begin() as conn:                                    # another program deletes the student
        conn.execute(delete(Enrollment).where(Enrollment.student_id == 23))
        conn.execute(delete(Student).where(Student.id == 23))

    wes.email = "wes.carter@college.edu"
    try:
        session.commit()
    except StaleDataError:
        session.rollback()
        print("not saved, and the student is now:", session.get(Student, 23))


not saved, and the student is now: None


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [24]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A relationship's default rule keeps the children and sets their foreign key to `NULL`, which
  suits a key that may be `NULL` and fails on one that may not.
- `cascade="all, delete-orphan"` deletes the children with their parent, and a child taken out of
  its collection, after a `SELECT` that loads them.
- `passive_deletes=True` with `ondelete="CASCADE"` leaves the children to the database: one
  `DELETE`, on a connection with foreign keys switched on.
- A bulk `delete()` is one statement that runs none of the ORM's rules, so only the database's
  apply.
- A cascade goes on the one-to-many side, and a row another program deleted first makes an `UPDATE`
  raise `StaleDataError` and a `DELETE` warn.


## What is next

The **Async SQLAlchemy** notebook runs sessions like these without blocking: `AsyncSession` and
`async_sessionmaker` on the `aiosqlite` driver, and the lazy load that an async session refuses.


---

&#8592; **Previous:** [Joins and Aggregates](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/14-joins-and-aggregates.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Async SQLAlchemy](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/16-async-sqlalchemy.ipynb) &#8594;
